# data_access

> Functionality to access Euclid calibrated frames, mosaic tiles and raw data for specific targets and observations.

In [ ]:
# | default_exp euclid.data_access

In [ ]:
# | export

import logging
import os
import re
import tempfile
import time
from datetime import datetime, timedelta
from getpass import getpass
from multiprocessing import Process
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from astropy import table
from astropy.io import fits
from astropy.table import Table
from astroquery.utils.tap.core import TapPlus

from nicl.euclid.utilities import (
    default_data_path,
    euclid_credentials,
    get_dither_id_from_filename,
)
from nicl import configure_logging
from nicl.utilities import maybe_to_value

In [ ]:
# | hide
from nbdev import show_doc

In [ ]:
# | exporti

configure_logging(level="INFO")
configure_logging(name=__name__, level="INFO")
logging.getLogger("astroquery").setLevel(logging.WARNING)

In [ ]:
# | export


class DataAccess:
    """Provides access to Euclid data."""

    def __init__(
        self,
        esa_username=None,  # ESA account username (prompts if not supplied)
        esa_password=None,  # ESA account password (prompts if not supplied)
        release_name="DR1",  # Q1 or DR1
        esac_server_url=None,  # ESA server (default is to choose based on release_name)
        dry_run=False,  # if True, do not actually download files
        overwrite=False,  # should existing files be overwritten?
        science_only=True,  # if True, only return science frames or observations containing them
    ):
        """Create an object for accessing data and log in to the ESA server."""
        credentials = euclid_credentials()
        if credentials is not None:
            if esa_username is None:
                esa_username = credentials["user"]
            if esa_password is None:
                esa_password = credentials["password"]
        if esa_username is None:
            esa_username = getpass(prompt="ESA User:")
        if esa_password is None:
            esa_password = getpass(prompt="ESA Password:")
        self.esa_username = esa_username
        self.esa_password = esa_password
        self.dry_run = dry_run
        self.overwrite = overwrite
        self.science_only = science_only
        if release_name.startswith("Q1"):
            esac_server_url = "https://eas.esac.esa.int"
            self.release_condition = f"(release_name='{release_name}')"
        elif release_name.startswith("DR1"):
            esac_server_url = "https://easidr.esac.esa.int"
            self.release_condition = f"(environment='{release_name}')"
        elif release_name.startswith("OTF"):
            esac_server_url = "https://easotf.esac.esa.int"
            self.release_condition = None
        self.tap = TapPlus(url=f"{esac_server_url}/tap-server", tap_context="tap")
        self.data_tap = TapPlus(url=f"{esac_server_url}/sas-dd", data_context="data")
        self.mer_filename_lookup = self.get_mer_filename_lookup()
        self._last_login_time = None
        self._last_data_login_time = None
        self.logger = logging.getLogger(__name__)

    def try_tap_login(self, tap, n_tries=3):
        attempt = 0
        while True:
            try:
                tap.login(user=self.esa_username, password=self.esa_password)
                break
            except TimeoutError as e:
                attempt += 1
                if attempt >= n_tries:
                    self.logger.warning(f"TAP login failed after {n_tries} attempts")
                    raise e
                delay = 5 + 5 * attempt
                self.logger.warning(f"TAP login timed out; retrying in {delay} seconds")
                time.sleep(delay)

    def tap_login(self):
        if self._last_login_time is None or time.time() - self._last_login_time > 60:
            self.try_tap_login(self.tap)
            self._last_login_time = time.time()

    def data_login(self):
        if (
            self._last_data_login_time is None
            or time.time() - self._last_data_login_time > 60
        ):
            self.try_tap_login(self.data_tap)
            self._last_data_login_time = time.time()

    def tap_query(
        self,
        query: str,  # ADQL query string
        *,
        retry_async_on_timeout: bool = True,  # If True, on sync timeout re-launch as async and wait up to 2h
        async_only: bool = False,  # If True, skip sync and run as async from the start (for long-running queries)
    ) -> Table:  # Query result table
        """Run a TAP query. Optionally retry with an async job if sync times out."""
        self.tap_login()
        if async_only:
            return self._tap_query_async_wait(query)
        if retry_async_on_timeout:
            try:
                job = self.tap.launch_job(query)
                results = job.get_results()
                if len(results) == 2000:
                    self.logger.warning("Query results may be truncated.")
                return results
            except requests.exceptions.Timeout as e:
                self.logger.warning(
                    "Synchronous TAP query timed out (%s); re-launching as asynchronous job (max wait 2h).",
                    e,
                )
                return self._tap_query_async_wait(query)
        job = self.tap.launch_job(query)
        results = job.get_results()
        if len(results) == 2000:
            self.logger.warning("Query results may be truncated.")
        return results

    def _tap_query_async_wait(self, query, max_wait_seconds=7200):
        """Run query as async job and wait until finished, polling with decreasing frequency."""
        job = self.tap.launch_job_async(query, background=True)
        interval = 5
        total_waited = 0
        while total_waited < max_wait_seconds:
            phase = job.get_phase(update=True).upper().strip()
            if phase in ("COMPLETED", "ERROR", "ABORTED"):
                break
            next_poll_at = datetime.now() + timedelta(seconds=interval)
            self.logger.info(
                "TAP async job still running (phase=%s). Next poll in %ds at %s.",
                phase,
                interval,
                next_poll_at.strftime("%Y-%m-%d %H:%M:%S"),
            )
            time.sleep(interval)
            total_waited += interval
            interval = min(300, interval + 5)
        if phase == "ERROR" or phase == "ABORTED":
            err = job.get_error()
            raise RuntimeError(f"TAP async job ended with phase {phase}: {err}")
        if total_waited >= max_wait_seconds:
            raise TimeoutError(
                f"TAP async job did not complete within {max_wait_seconds}s (2h)."
            )
        results = job.get_results()
        if len(results) == 2000:
            self.logger.warning("Query results may be truncated.")
        return results

    def get_mer_filename_lookup(self):
        """Build a DataFrame of MER files.

        This assumes there is a file named "filelist" in the MER folder, containing
        a listing of the MER files in archive, created by running `find .` in the
        MER folder in DataLabs. This is a workaround, due to the normal ways of
        accessing non-STK MER files not currently working.
        """

        def get_mer_info(row):
            fn = row["filename"]
            matched = re.search("EUC_MER_(.+)_TILE", fn)[1]
            if matched.endswith("RMS") or matched.endswith("FLAG"):
                match = re.search("MOSAIC-(.+)-(RMS|FLAG)", matched)
                filt, file_type = match.groups()
            else:
                match = re.search("(.+)-(.+-?.)", matched)
                file_type, filt = match.groups()
            if file_type == "BGSUB-MOSAIC":
                t = "STK"
            elif file_type == "BGMOD":
                t = "BKG"
            else:
                t = file_type
            filt = filt.replace("-", "_")
            return t, filt

        path = default_data_path("Q1_R1", "MER")
        fn = path / "filelist"
        if fn.exists():
            table = pd.read_table(
                fn,
                header=None,
                names=["dot", "tile_index", "instrument", "filename"],
                dtype=str,
                sep="/",
            )
            table = table.drop(columns="dot").dropna()
            table = table[table["filename"].str.startswith("EUC")]
            cols = table.apply(get_mer_info, axis="columns", result_type="expand")
            cols.columns = ["mer_file_type", "filter"]
            table = pd.concat((table, cols), axis="columns")
            table = table.groupby(
                ["tile_index", "instrument", "mer_file_type", "filter"]
            ).first()
            table = table["filename"]
            return table

    def build_instrument_condition(self, instrument, filter, raw=False):
        conditions = []
        if self.release_condition and not raw:
            conditions.append(self.release_condition)
        if instrument is not None:
            conditions.append(f"(instrument_name = '{instrument}')")
        if filter is not None:
            conditions.append(f"(filter_name = '{filter}')")
        return " AND ".join(conditions)

    def build_fov_condition(self, ra, dec, radius, fully_contained):
        ra, dec, radius = (maybe_to_value(x, "deg") for x in (ra, dec, radius))
        conditions = []
        if self.release_condition:
            conditions.append(self.release_condition)
        criterion = "CONTAINS" if fully_contained else "INTERSECTS"
        conditions.append(
            f"(fov IS NOT NULL AND {criterion}(CIRCLE('ICRS',{ra},{dec},{radius}),fov)=1)"
        )
        return " AND ".join(conditions)

    def find_all_observations(
        self,
    ):  # returns a list of observation_ids
        """Obtain a list of all survey obs_ids for observations in the current release."""
        query = """SELECT DISTINCT TOP 1000000 observation_id
                    FROM sedm.calibrated_frame
                    WHERE (product_type like '%Calibrated%')\n"""
        if self.science_only:
            query += "AND (category = 'SCIENCE')\n"
        if self.release_condition:
            query += f"AND {self.release_condition}\n"
        query += "ORDER BY observation_id ASC\n"
        results = self.tap_query(query)
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def find_all_tiles(
        self,
    ):  # returns a list of observation_ids
        """Obtain a list of all MER tile_indexes for tiles in the current release."""
        query = """SELECT DISTINCT TOP 1000000 tile_index
                    FROM sedm.mosaic_product\n"""
        if self.release_condition:
            query += f"WHERE {self.release_condition}\n"
        query += "ORDER BY tile_index ASC"
        results = self.tap_query(query)
        tile_indexes = np.unique(list(results["tile_index"])).astype(int)
        return tile_indexes

    def find_observations_for_target(
        self,
        ra,  # RA of the target, as an angular quantity or in decimal degrees
        dec,  # Dec of the target, as an angular quantity or in decimal degrees
        radius=1
        / 60,  # radius of the target, as an angular quantity or in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of observation_ids
        """Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region."""
        condition = self.build_fov_condition(ra, dec, radius, fully_contained)
        query = """SELECT DISTINCT observation_id
                   FROM sedm.calibrated_frame
                   WHERE (product_type like '%Calibrated%')\n"""
        if self.science_only:
            query += "AND (category = 'SCIENCE')\n"
        query += f"AND {condition}\n"
        query += "ORDER BY observation_id ASC"
        results = self.tap_query(query)
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def find_tiles_for_target(
        self,
        ra,  # RA of the target, as an angular quantity or in decimal degrees
        dec,  # Dec of the target, as an angular quantity or in decimal degrees
        radius=1
        / 60,  # radius of the target, as an angular quantity or in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of tile_indexes
        """Obtain a list of survey MER tile_indexes for tiles that entirely contain or intersect the specified target region."""
        condition = self.build_fov_condition(ra, dec, radius, fully_contained)
        query = f"""SELECT DISTINCT tile_index
                    FROM sedm.mosaic_product
                    WHERE {condition}
                    ORDER BY tile_index ASC"""
        results = self.tap_query(query)
        tile_indexes = np.unique(list(results["tile_index"])).astype(int)
        return tile_indexes

    def get_calibrated_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain calibrated file information for obs_id, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter)
        query = """SELECT observation_stack.observation_id, observation_stack.file_name,
                   observation_stack.instrument_name, observation_stack.filter_name, observation_stack.duration
                   FROM sedm.calibrated_frame
                   AS observation_stack
                   WHERE (product_type like '%Calibrated%')\n"""
        if self.science_only:
            query += "AND (category = 'SCIENCE')\n"
        query += f"AND {condition}\n"
        query += f"AND (observation_id = '{obs_id}')"
        file_info = self.tap_query(query)
        return file_info

    def get_sir_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
    ):  # returns a table of file information
        """Obtain SIR file information for obs_id."""
        query = f"""SELECT observation_id, file_name
                    FROM sedm.sir_science_frame
                    WHERE (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        return file_info

    def get_raw_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, "" for VIS, OPEN for grism, CLOSED for slew dark, Y, J or H
    ):  # returns a table of file information
        """Obtain raw file information for obs_id, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter, raw=True)
        query = f"""SELECT raw_frame.file_name, raw_frame.rawframe_oid, raw_frame.observation_id,
                    raw_frame.instrument_name, raw_frame.data_set_release, raw_frame.filter_name,
                    raw_frame.observation_mode, raw_frame.grism_wheel_pos, raw_frame.cal_block_id,
                    raw_frame.cal_block_variant, raw_frame.ra, raw_frame.dec, raw_frame.obs_time_utc,
                    raw_frame.exposure_time
                    FROM sedm.raw_frame
                    WHERE {condition}
                    AND (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        return file_info

    def get_files_for_tile(
        self,
        tile_index,  # tile_index for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain mosaic file information for tile_index, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter)
        query = f"""SELECT mosaic_product.tile_index, mosaic_product.file_name, mosaic_product.mosaic_product_oid,
                    mosaic_product.instrument_name, mosaic_product.filter_name, mosaic_product.category,
                    mosaic_product.second_type, mosaic_product.ra, mosaic_product.dec, mosaic_product.technique 
                    FROM sedm.mosaic_product
                    WHERE {condition}
                    AND (tile_index = '{tile_index}')"""
        file_info = self.tap_query(query)
        return file_info

    def download_files(
        self,
        filenames,  #  list of filenames or Table containing "file_name" column
        outpath,  # the folder in which to save the downloaded files
    ):
        """Download multiple Euclid filenames to outpath."""
        outpath = Path(outpath).expanduser()
        outpath.mkdir(parents=True, exist_ok=True)
        if isinstance(filenames, Table):
            filenames = filenames["file_name"]
        self.logger.info(f"Downloading {len(filenames)} files to {outpath}")
        for fn in filenames:
            self.logger.info(f"Downloading {fn}")
            self.download_file(fn, outpath)
            self.check_file_size(fn, outpath)

    def check_file_size(self, fn, outpath):
        "Check the size of a file matches expectations."
        if fn.startswith("EUC_VIS_SWL-DET"):
            expected_size = 7315678080
        elif fn.startswith("EUC_NIR_W-CAL-IMAGE"):
            expected_size = 799361280
        elif fn.startswith("EUC_SIR_W-SCIFRM_BKGSUB"):
            expected_size = 799266240
        else:
            expected_size = None
        actual_size = os.path.getsize(outpath / fn)
        if expected_size is not None and actual_size != expected_size:
            self.logger.warning(
                f"File {fn}: size {actual_size}, but expected {expected_size}"
            )

    def _download_worker(self, params, output_path):
        "Download a file; used by _download_file_with_timeout in a separate process."
        self.data_login()
        self.data_tap.load_data(params_dict=params, output_file=output_path)

    def _download_file_with_timeout(
        self,
        params_dict,
        outfn,
        stall_timeout_seconds=3600,  # 1 hour
        check_interval_seconds=60,
        max_retries=3,
    ):
        """Download a file with a timeout.

        This is a workaround for the fact that the TAP service does not support timeouts on file downloads.
        If the download stalls, the process is killed and retried.

        """
        for attempt in range(1, max_retries + 1):
            if outfn.exists():
                os.remove(outfn)

            proc = Process(target=self._download_worker, args=(params_dict, outfn))
            proc.start()

            last_size = outfn.stat().st_size if outfn.exists() else 0
            last_change = time.monotonic()

            while proc.is_alive():
                time.sleep(check_interval_seconds)
                self.logger.debug("Checking download progress")
                size = outfn.stat().st_size if outfn.exists() else 0
                self.logger.debug(f"Downloaded {size / 1024 / 1024:.2f} Mb")

                if size > last_size:
                    last_size = size
                    last_change = time.monotonic()
                elif time.monotonic() - last_change > stall_timeout_seconds:
                    self.logger.warning(
                        f"Download of {outfn} stalled for {stall_timeout_seconds} s; "
                        f"killing and retrying (attempt {attempt}/{max_retries})."
                    )
                    proc.terminate()
                    proc.join()
                    if outfn.exists():
                        os.remove(outfn)
                    break  # retry outer loop

            else:
                # process exited normally
                proc.join()
                return

        raise TimeoutError(
            f"Download of {outfn} stalled and failed after {max_retries} attempts."
        )

    def download_file(
        self,
        filename,  # the filename to download
        outpath,  # the folder in which to save the downloaded files
        ignore_dry_run=False,  # ignore object's dry_run setting
    ):
        """Download Euclid filename to outpath."""
        params_dict = dict(
            RETRIEVAL_TYPE="FILE", RELEASE="sedm", RETRIEVAL_ACCESS="DIRECT"
        )
        params_dict.update(FILE_NAME=filename)
        outpath = Path(outpath).expanduser()
        outfn = outpath / filename
        if ignore_dry_run or not self.dry_run:
            if not outfn.exists() or self.overwrite:
                try:
                    self._download_file_with_timeout(
                        params_dict=params_dict, outfn=outfn
                    )
                except (Exception, KeyboardInterrupt):
                    if outfn.exists():
                        os.remove(outfn)
                    self.logger.warning(f"Error downloading {filename}")
            else:
                self.logger.info(f"File already exists, skipping: {outfn}")

    def download_mosaic_files(
        self,
        file_info,  #  Table containing file information
        outpath,  # the folder in which to save the downloaded files
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
    ):
        """Download multiple Euclid mosaics to outpath."""
        outpath = Path(outpath).expanduser()
        outpath.mkdir(parents=True, exist_ok=True)
        self.logger.info(f"Downloading {len(file_info)} files to {outpath}")
        filenames = []
        for info in file_info:
            t = info["tile_index"]
            i = info["instrument_name"]
            f = info["filter_name"]
            fn = self.download_mosaic_file(t, i, f, outpath, mer_file_type)
            filenames.append(fn)
        return filenames

    def download_mosaic_file(
        self,
        tile_index,
        instrument,  # NISP or VIS
        filter,  # VIS, NIR_Y, NIR_J or NIR_H
        outpath,  # the folder in which to save the downloaded files
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
    ):
        """Download Euclid mosaic to outpath, using a workaround method."""
        if self.mer_filename_lookup is None:
            raise FileNotFoundError("MER filename lookup file was not found.")
        try:
            fn = self.mer_filename_lookup.loc[
                f"{tile_index}", instrument, mer_file_type, filter
            ]
        except KeyError:
            self.logger.warning(
                f"No MER file found in lookup table for tile {tile_index}, instrument {instrument} and filter {filter}."
            )
            return ""
        self.logger.info(f"Downloading {fn}")
        self.download_file(fn, outpath)
        return fn

    def download_mosaic_file_alt(
        self,
        tile_index,
        instrument,  # NISP or VIS
        filter,  # VIS, NIR_Y, NIR_J or NIR_H
        outpath,  # the folder in which to save the downloaded files
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
    ):
        """Download Euclid mosaic to outpath, using an alternative method.

        This is currently not supported by SAS for file types other than STK.
        """
        params_dict = dict(RETRIEVAL_TYPE="MOSAIC", RELEASE="sedm")
        params_dict.update(
            TILE_INDEX=tile_index,
            INSTRUMENT=instrument,
            FILTER=filter,
            TYPE=mer_file_type,
        )
        outpath = Path(outpath).expanduser()
        if mer_file_type == "STK":
            filename = f"BGSUB-MOSAIC-{filter}"
        elif mer_file_type == "BKG":
            filename = f"BGMOD-{filter}"
        elif mer_file_type == "RMS":
            filename = f"MOSAIC-{filter}-RMS"
        elif mer_file_type == "FLAG":
            filename = f"MOSAIC-{filter}-FLAG"
        else:
            raise ValueError(f"Invalid mer_file_type provided: {mer_file_type}.")
        filename = f"EUC_MER_{filename}_TILE{tile_index}.fits"
        outfn = outpath / filename
        self.logger.info(
            f"Downloading {mer_file_type} file for tile {tile_index}, instrument {instrument} and filter {filter}"
        )
        self.logger.info(f"Saving as {outfn}")
        if not self.dry_run:
            if not outfn.exists() or self.overwrite:
                self.data_login()
                self.data_tap.load_data(params_dict=params_dict, output_file=outfn)
            else:
                self.logger.info(f"File already exists, skipping: {outfn}")

    def download_calibrated_files_for_observation(
        self,
        obs_id,
        outpath,  # the base folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        include_vis_short=False,  # if True, include VIS short exposure files
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_calibrated_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        if instrument == "VIS" and not include_vis_short:
            file_info = file_info[file_info["duration"] > 300]
        if instrument == "NISP":
            instrument_folder = "NIR"
        elif instrument == "VIS":
            instrument_folder = "VIS_QUAD"
        else:
            raise ValueError("The instrument must be NISP or VIS.")
        outpath = Path(outpath, instrument_folder, f"{obs_id:n}")
        self.download_files(file_info, outpath=outpath)
        return file_info

    def download_sir_files_for_observation(
        self,
        obs_id,
        outpath,  # the base folder in which to save the downloaded files
    ):  #  returns a table of file information
        """Download all SIR files for a Euclid observation."""
        file_info = self.get_sir_files_for_observation(obs_id)
        outpath = Path(outpath, "SIR", f"{obs_id:n}")
        self.download_files(file_info, outpath=outpath)
        return file_info

    def download_raw_files_for_observation(
        self,
        obs_id,
        outpath,  # the base folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_raw_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        outpath = Path(outpath, "RAW", f"{obs_id:n}")
        self.download_files(file_info, outpath=outpath)
        return file_info

    def download_files_for_tile(
        self,
        tile_index,
        outpath,  # the base folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
    ):  #  returns a table of file information
        """Download all files for a Euclid MER tile, optionally restricted by instrument or filter.

        By default this gets the background subtracted image, but specifying `mer_file_type` can download the
        background model (BKG), RMS or FLAG images.
        """
        file_info = self.get_files_for_tile(
            tile_index, instrument=instrument, filter=filter
        )
        if instrument == "NISP":
            instrument_folder = "NIR"
        elif instrument == "VIS":
            instrument_folder = "VIS"
        else:
            raise ValueError("The instrument must be NISP or VIS.")
        outpath = Path(outpath, "MER", f"{tile_index:n}", instrument_folder)
        filenames = self.download_mosaic_files(
            file_info, outpath=outpath, mer_file_type=mer_file_type
        )
        file_info["file_name"] = filenames
        return file_info

    def download_files_for_target(
        self,
        ra,  # RA of the target, as an angular quantity or in decimal degrees
        dec,  # Dec of the target, as an angular quantity or in decimal degrees
        radius=1
        / 60,  # radius of the target, as an angular quantity or in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
        outpath=None,  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        file_type="CAL",  # CAL, MER
        mer_file_type="STK",  # STK, BKG, RMS or FLAG, only for file_type="MER"
    ):  #  returns a table of file information
        """Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter."""
        file_info = []
        if file_type == "CAL":
            obs_ids = self.find_observations_for_target(
                ra, dec, radius, fully_contained=fully_contained
            )
            for obs_id in obs_ids:
                self.logger.info(f"Downloading files for observation id {obs_id}")
                obs_file_info = self.download_calibrated_files_for_observation(
                    obs_id,
                    outpath=outpath,
                    instrument=instrument,
                    filter=filter,
                )
                file_info.append(obs_file_info)
        elif file_type == "MER":
            tile_ids = self.find_tiles_for_target(
                ra, dec, radius, fully_contained=fully_contained
            )
            for tile_id in tile_ids:
                self.logger.info(f"Downloading files for tile index {tile_id}")
                tile_file_info = self.download_files_for_tile(
                    tile_id,
                    outpath=outpath,
                    instrument=instrument,
                    filter=filter,
                    mer_file_type=mer_file_type,
                )
                file_info.append(tile_file_info)
        if len(file_info) > 0:
            return table.vstack(file_info)

    def get_catalogues_for_observation(
        self,
        obs_id,
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        dither=None,  # 0, 1, 2, 3 for NISP, 0-1, 1-1, 2-1, 3-1, 0-2, 1-2 for VIS
    ):  #  returns a squashed dictionary containing one catalogue per filter and dither
        """Download all catalogues for a Euclid observation, optionally restricted by instrument or filter."""
        if instrument == "VIS":
            filter = "VIS"
        condition = self.build_instrument_condition(instrument, filter)
        query = f"""SELECT observation_id, file_name, instrument_name, filter_name
            FROM sedm.frame_catalog
            WHERE (product_type like '%Calibrated%')
            AND {condition}
            AND (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        catalogues = dict()
        with tempfile.TemporaryDirectory() as temp_dir:
            for row in file_info:
                fn = row["file_name"]
                this_filter = str(row["filter_name"])
                this_dither = get_dither_id_from_filename(fn)
                if dither is None or this_dither == dither:
                    self.download_file(fn, outpath=temp_dir, ignore_dry_run=True)
                    cat_file = os.path.join(temp_dir, fn)
                    with fits.open(cat_file) as hdul:
                        cats = []
                        for hdu in hdul:
                            if hdu.name == "LDAC_OBJECTS":
                                tab = Table.read(hdu)
                                if this_filter == "VIS":
                                    tab["DETECTOR"] = (
                                        f"{tab.meta['CCDID']}.{tab.meta['QUADID']}"
                                    )
                                else:
                                    tab["DETECTOR"] = tab.meta["DET_ID"]
                                tab.meta = None
                                cats.append(tab)
                        if this_filter not in catalogues:
                            catalogues[this_filter] = dict()
                        catalogues[this_filter][this_dither] = table.vstack(cats)
        if dither:
            for filt in catalogues:
                catalogues[filt] = catalogues[filt][dither]
        if filter:
            catalogues = catalogues[filter]
        return catalogues

### Calibrated files

In [ ]:
show_doc(DataAccess.find_observations_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L260){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.find_observations_for_target

```python

def find_observations_for_target(
    ra, # RA of the target, as an angular quantity or in decimal degrees
    dec, # Dec of the target, as an angular quantity or in decimal degrees
    radius:float=0.016666666666666666,
    fully_contained:bool=True, # if False, the target region only needs to intersect with the observation footprint
):


```

*Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region.*

In [ ]:
show_doc(DataAccess.get_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L299){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_calibrated_files_for_observation

```python

def get_calibrated_files_for_observation(
    obs_id, # observation_id for which to find files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, VIS, NIR_Y, NIR_J or NIR_H
):


```

*Obtain calibrated file information for obs_id, optionally restricted by instrument or filter.*

In [ ]:
show_doc(DataAccess.download_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L492){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_calibrated_files_for_observation

```python

def download_calibrated_files_for_observation(
    obs_id, outpath, # the base folder in which to save the downloaded files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, VIS, NIR_Y, NIR_J or NIR_H
    include_vis_short:bool=False, # if True, include VIS short exposure files
):


```

*Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter.*

In [ ]:
show_doc(DataAccess.download_files_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L571){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_files_for_target

```python

def download_files_for_target(
    ra, # RA of the target, as an angular quantity or in decimal degrees
    dec, # Dec of the target, as an angular quantity or in decimal degrees
    radius:float=0.016666666666666666,
    fully_contained:bool=True, # if False, the target region only needs to intersect with the observation footprint
    outpath:NoneType=None, # the folder in which to save the downloaded files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, VIS, NIR_Y, NIR_J or NIR_H
    file_type:str='CAL', # CAL, MER
    mer_file_type:str='STK', # STK, BKG, RMS or FLAG, only for file_type="MER"
):


```

*Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter.*

### Mosaic files

In [ ]:
show_doc(DataAccess.find_tiles_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L281){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.find_tiles_for_target

```python

def find_tiles_for_target(
    ra, # RA of the target, as an angular quantity or in decimal degrees
    dec, # Dec of the target, as an angular quantity or in decimal degrees
    radius:float=0.016666666666666666,
    fully_contained:bool=True, # if False, the target region only needs to intersect with the observation footprint
):


```

*Obtain a list of survey MER tile_indexes for tiles that entirely contain or intersect the specified target region.*

In [ ]:
show_doc(DataAccess.get_files_for_tile)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L349){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_files_for_tile

```python

def get_files_for_tile(
    tile_index, # tile_index for which to find files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, VIS, NIR_Y, NIR_J or NIR_H
):


```

*Obtain mosaic file information for tile_index, optionally restricted by instrument or filter.*

In [ ]:
show_doc(DataAccess.download_files_for_tile)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L542){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_files_for_tile

```python

def download_files_for_tile(
    tile_index, outpath, # the base folder in which to save the downloaded files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, VIS, NIR_Y, NIR_J or NIR_H
    mer_file_type:str='STK', # STK, BKG, RMS or FLAG
):


```

*Download all files for a Euclid MER tile, optionally restricted by instrument or filter.*

By default this gets the background subtracted image, but specifying `mer_file_type` can download the
background model (BKG), RMS or FLAG images.

### SIR files

In [ ]:
show_doc(DataAccess.get_sir_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L319){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_sir_files_for_observation

```python

def get_sir_files_for_observation(
    obs_id, # observation_id for which to find files
):


```

*Obtain SIR file information for obs_id.*

In [ ]:
show_doc(DataAccess.download_sir_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L516){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_sir_files_for_observation

```python

def download_sir_files_for_observation(
    obs_id, outpath, # the base folder in which to save the downloaded files
):


```

*Download all SIR files for a Euclid observation.*

### Raw files

In [ ]:
show_doc(DataAccess.get_raw_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L330){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_raw_files_for_observation

```python

def get_raw_files_for_observation(
    obs_id, # observation_id for which to find files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, "" for VIS, OPEN for grism, CLOSED for slew dark, Y, J or H
):


```

*Obtain raw file information for obs_id, optionally restricted by instrument or filter.*

In [ ]:
show_doc(DataAccess.download_raw_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L527){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_raw_files_for_observation

```python

def download_raw_files_for_observation(
    obs_id, outpath, # the base folder in which to save the downloaded files
    instrument:NoneType=None, # None, NISP or VIS
    filter:NoneType=None, # None, VIS, NIR_Y, NIR_J or NIR_H
):


```

*Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter.*

### Example

The following demonstrates use of the DataAccess class. This would normally first be imported using `from nicl.euclid.data_access import DataAccess`.

To actually download files, remove `dry_run=True`.

In [ ]:
release_name = "DR1"

da = DataAccess(release_name=release_name, dry_run=True)

We will download the files into the folder structure at the default location.

In [ ]:
outpath = default_data_path(release_name)

#### Calibrated frames

In [ ]:
da.find_all_observations()

array([ 10, 100, 101, ...,  97,  98,  99], shape=(6458,))

In [ ]:
da.find_observations_for_target(ra=268.6647, dec=68.0564)

array([2036, 2080, 2081, 2195, 2683, 3009, 3318, 5025, 5586, 6149, 6895,
       7777])

In [ ]:
da.find_observations_for_target(
    ra=268.6647, dec=68.0564, radius=1.0, fully_contained=False
)

array([2036, 2037, 2038, 2040, 2041, 2042, 2043, 2045, 2046, 2047, 2079,
       2080, 2081, 2082, 2083, 2084, 2085, 2090, 2091, 2092, 2194, 2195,
       2196, 2198, 2199, 2200, 2201, 2205, 2682, 2683, 2684, 2685, 2686,
       2693, 2694, 2695, 2995, 2996, 3004, 3005, 3006, 3007, 3008, 3009,
       3010, 3011, 3307, 3308, 3309, 3317, 3318, 3319, 3320, 3321, 3333,
       5009, 5010, 5014, 5015, 5016, 5017, 5024, 5025, 5026, 5578, 5579,
       5580, 5585, 5586, 5587, 6142, 6143, 6147, 6148, 6149, 6150, 6151,
       6152, 6162, 6163, 6694, 6696, 6697, 6879, 6880, 6881, 6882, 6883,
       6884, 6894, 6895, 6896, 6897, 7770, 7776, 7777, 7778, 7779, 7780,
       7790, 7791])

In [ ]:
da.get_calibrated_files_for_observation(2683, instrument="NISP", filter="NIR_H")

observation_id,file_name,instrument_name,filter_name,duration
object,object,object,object,float64
2683,EUC_NIR_W-CAL-IMAGE_H-2683-0_20250615T170134.034528Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-1_20250615T170134.639919Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-3_20250615T170133.971277Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-2_20250615T170133.996269Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_observation(
    2683, instrument="NISP", filter="NIR_H", outpath=outpath
)

2026-02-27 10:57:59 - __main__.download_files - INFO - Downloading 4 files to /home/ppzsb1/euclid_data/DR1/NIR/2683
2026-02-27 10:57:59 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2683-0_20250615T170134.034528Z.fits
2026-02-27 10:57:59 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2683-3_20250615T170133.971277Z.fits
2026-02-27 10:57:59 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2683-2_20250615T170133.996269Z.fits
2026-02-27 10:57:59 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2683-1_20250615T170134.639919Z.fits


observation_id,file_name,instrument_name,filter_name,duration
object,object,object,object,float64
2683,EUC_NIR_W-CAL-IMAGE_H-2683-0_20250615T170134.034528Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-3_20250615T170133.971277Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-2_20250615T170133.996269Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-1_20250615T170134.639919Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_files_for_target(
    ra=268.6647,
    dec=68.0564,
    instrument="NISP",
    filter="NIR_H",
    file_type="CAL",
    outpath=outpath,
)

2026-02-27 10:58:00 - __main__.download_files_for_target - INFO - Downloading files for observation id 2036
2026-02-27 10:58:01 - __main__.download_files - INFO - Downloading 4 files to /home/ppzsb1/euclid_data/DR1/NIR/2036
2026-02-27 10:58:01 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2036-1_20250615T225801.571332Z.fits
2026-02-27 10:58:01 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2036-0_20250615T225801.606367Z.fits
2026-02-27 10:58:01 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2036-3_20250615T225801.509846Z.fits
2026-02-27 10:58:01 - __main__.download_files - INFO - Downloading EUC_NIR_W-CAL-IMAGE_H-2036-2_20250615T225801.601566Z.fits
2026-02-27 10:58:01 - __main__.download_files_for_target - INFO - Downloading files for observation id 2080
2026-02-27 10:58:01 - __main__.download_files - INFO - Downloading 4 files to /home/ppzsb1/euclid_data/DR1/NIR/2080
2026-02-27 10:58:01 - __main__.download_files - INFO

observation_id,file_name,instrument_name,filter_name,duration
object,object,object,object,float64
2036,EUC_NIR_W-CAL-IMAGE_H-2036-1_20250615T225801.571332Z.fits,NISP,NIR_H,87.2448
2036,EUC_NIR_W-CAL-IMAGE_H-2036-0_20250615T225801.606367Z.fits,NISP,NIR_H,87.2448
2036,EUC_NIR_W-CAL-IMAGE_H-2036-3_20250615T225801.509846Z.fits,NISP,NIR_H,87.2448
2036,EUC_NIR_W-CAL-IMAGE_H-2036-2_20250615T225801.601566Z.fits,NISP,NIR_H,87.2448
2080,EUC_NIR_W-CAL-IMAGE_H-2080-0_20250616T100524.064354Z.fits,NISP,NIR_H,87.2448
2080,EUC_NIR_W-CAL-IMAGE_H-2080-2_20250616T100524.458744Z.fits,NISP,NIR_H,87.2448
2080,EUC_NIR_W-CAL-IMAGE_H-2080-3_20250616T100524.084071Z.fits,NISP,NIR_H,87.2448
2080,EUC_NIR_W-CAL-IMAGE_H-2080-1_20250616T100524.102918Z.fits,NISP,NIR_H,87.2448
2081,EUC_NIR_W-CAL-IMAGE_H-2081-2_20250616T060009.774537Z.fits,NISP,NIR_H,87.2448


#### MER tiles

::: {.callout-warning}
Note that currently (at December 2024) the documented methods to programatically download the MER auxilliary files (e.g., BKG, RMS, FLAG) are broken. Instead, a workaround has been implemented, which requires a file listing of the data archive to be first obtained. This can be created by running `find .` in the MER folder in DataLabs.
:::

In [ ]:
da.find_all_tiles()

array([101271287, 101272183, 101272184, ..., 102165066, 102165078,
       102165079], shape=(10991,))

In [ ]:
da.find_tiles_for_target(ra=268.6647, dec=68.0564)

array([101838582, 102160336])

In [ ]:
da.find_tiles_for_target(ra=268.6647, dec=68.0564, radius=1.0, fully_contained=False)

array([101836356, 101836357, 101836358, 101836359, 101836360, 101836920,
       101836921, 101836922, 101836923, 101836924, 101836925, 101836926,
       101837478, 101837479, 101837480, 101837481, 101837482, 101837483,
       101837484, 101837485, 101838030, 101838031, 101838032, 101838033,
       101838034, 101838035, 101838036, 101838037, 101838038, 101838578,
       101838579, 101838580, 101838581, 101838582, 101838583, 101838584,
       101838585, 101838586, 101839119, 101839120, 101839121, 101839122,
       101839123, 101839124, 101839125, 101839126, 101839127, 101839128,
       101839655, 101839656, 101839657, 101839658, 101839659, 101839660,
       101839661, 101839662, 101839663, 101840185, 101840186, 101840187,
       101840188, 101840189, 101840190, 101840191, 101840192, 101840710,
       101840712, 101840713, 101840714, 101840715, 102159773, 102159774,
       102159775, 102159776, 102160056, 102160057, 102160058, 102160059,
       102160060, 102160334, 102160335, 102160336, 

In [ ]:
da.get_files_for_tile(102160336, instrument="NISP", filter="NIR_H")

tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
,,,,,,,deg,deg,
int64,object,int64,object,object,object,object,float64,float64,object
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-2D014C_20250717T175714.254968Z_00.00.fits,85012,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-6CDE8D_20250805T041942.325380Z_00.00.fits,60321,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-120000_20250725T002509.891257Z_00.00.fits,19220,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-FC75FE_20250711T191834.850629Z_00.00.fits,76628,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D2D780_20250716T222803.966785Z_00.00.fits,85538,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-F52D83_20250725T152242.639588Z_00.00.fits,19473,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-949845_20250720T195656.079240Z_00.00.fits,13465,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-30E94C_20250705T221901.412670Z_00.00.fits,8373,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_tile(
    102160336, instrument="NISP", filter="NIR_H", outpath=outpath
)

2026-02-27 10:58:13 - __main__.download_mosaic_files - INFO - Downloading 11 files to /home/ppzsb1/euclid_data/DR1/MER/102160336/NIR
2026-02-27 10:58:13 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:13 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:13 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:13 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:13 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:13 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-

tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
,,,,,,,deg,deg,
int64,str82,int64,object,object,object,object,float64,float64,object
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,6757,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,34609,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,85538,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,60321,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,8373,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,76628,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,19473,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,13465,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_tile(
    102160336, instrument="NISP", filter="NIR_H", outpath=outpath, mer_file_type="BKG"
)

2026-02-27 10:58:14 - __main__.download_mosaic_files - INFO - Downloading 11 files to /home/ppzsb1/euclid_data/DR1/MER/102160336/NIR
2026-02-27 10:58:14 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits
2026-02-27 10:58:14 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits
2026-02-27 10:58:14 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits
2026-02-27 10:58:14 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits
2026-02-27 10:58:14 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits
2026-02-27 10:58:14 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024

tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
,,,,,,,deg,deg,
int64,str75,int64,object,object,object,object,float64,float64,object
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,76628,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,60321,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,34609,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,19220,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,85012,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,8373,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,13465,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,18349,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_tile(
    102160336, instrument="NISP", filter="NIR_H", outpath=outpath, mer_file_type="RMS"
)

2026-02-27 10:58:15 - __main__.download_mosaic_files - INFO - Downloading 11 files to /home/ppzsb1/euclid_data/DR1/MER/102160336/NIR
2026-02-27 10:58:15 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits
2026-02-27 10:58:15 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits
2026-02-27 10:58:15 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits
2026-02-27 10:58:15 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits
2026-02-27 10:58:15 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits
2026-02-27 10:58:15 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_MOSAIC-NIR-H-RMS

tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
,,,,,,,deg,deg,
int64,str80,int64,object,object,object,object,float64,float64,object
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,18349,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,34609,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,76628,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,85538,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,6757,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,60321,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,85012,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,19220,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_target(
    ra=268.6647,
    dec=68.0564,
    instrument="NISP",
    filter="NIR_H",
    file_type="MER",
    outpath=outpath,
)

2026-02-27 10:58:15 - __main__.download_files_for_target - INFO - Downloading files for tile index 101838582
2026-02-27 10:58:16 - __main__.download_mosaic_files - INFO - Downloading 1 files to /home/ppzsb1/euclid_data/DR1/MER/101838582/NIR
2026-02-27 10:58:16 - __main__.download_mosaic_file - WARNING - No MER file found in lookup table for tile 101838582, instrument NISP and filter NIR_H.
2026-02-27 10:58:16 - __main__.download_files_for_target - INFO - Downloading files for tile index 102160336
2026-02-27 10:58:17 - __main__.download_mosaic_files - INFO - Downloading 11 files to /home/ppzsb1/euclid_data/DR1/MER/102160336/NIR
2026-02-27 10:58:17 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:17 - __main__.download_mosaic_file - INFO - Downloading EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits
2026-02-27 10:58:17 - __main__.download_mosaic_file -

tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
,,,,,,,deg,deg,
int64,str82,int64,object,object,object,object,float64,float64,object
101838582,,131352,NISP,NIR_H,SCIENCE,SKY,268.7187945157,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,34609,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,76628,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,19220,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,60321,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,8373,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,19473,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,13465,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


#### SIR frames

These are the calibrated, but not stacked, spectroscopic frames, which are used in the persistence correction.

In [ ]:
da.get_sir_files_for_observation(2683)

observation_id,file_name
object,object
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_20250904T031320.229831Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_20250904T031206.552356Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_20250904T030820.486308Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_20250809T054439.972766Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_20250809T054919.927647Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_20250904T030826.624500Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_20250809T054909.778044Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_20250809T054401.638056Z.fits


In [ ]:
da.download_sir_files_for_observation(2683, outpath=outpath)

2026-02-27 10:58:18 - __main__.download_files - INFO - Downloading 8 files to /home/ppzsb1/euclid_data/DR1/SIR/2683
2026-02-27 10:58:18 - __main__.download_files - INFO - Downloading EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_20250809T054439.972766Z.fits
2026-02-27 10:58:19 - __main__.download_files - INFO - Downloading EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_20250904T031320.229831Z.fits
2026-02-27 10:58:19 - __main__.download_files - INFO - Downloading EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_20250904T030826.624500Z.fits
2026-02-27 10:58:19 - __main__.download_files - INFO - Downloading EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_20250809T054919.927647Z.fits
2026-02-27 10:58:19 - __main__.download_files - INFO - Downloading EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_20250904T031206.552356Z.fits
2026-02-27 10:58:19 - __main__.download_files - INFO - Downloading EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_20250904T030820.486308Z.fits
2026-02-27 10:58:19 - __main__

observation_id,file_name
object,object
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_20250809T054439.972766Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_20250904T031320.229831Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_20250904T030826.624500Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_20250809T054919.927647Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_20250904T031206.552356Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_20250904T030820.486308Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_20250809T054909.778044Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_20250809T054401.638056Z.fits


#### Raw frames

For example, we can download the raw slew darks.

In [ ]:
da.get_raw_files_for_observation(2683, instrument="NISP", filter="CLOSED")

file_name,rawframe_oid,observation_id,instrument_name,data_set_release,filter_name,observation_mode,grism_wheel_pos,cal_block_id,cal_block_variant,ra,dec,obs_time_utc,exposure_time
,,,,,,,,,,deg,deg,,
object,int64,int64,object,object,object,object,object,object,object,float64,float64,object,float64


In [ ]:
da.download_raw_files_for_observation(
    2683, instrument="NISP", filter="CLOSED", outpath=outpath
)

2026-02-27 10:58:20 - __main__.download_files - INFO - Downloading 0 files to /home/ppzsb1/euclid_data/DR1/RAW/2683


file_name,rawframe_oid,observation_id,instrument_name,data_set_release,filter_name,observation_mode,grism_wheel_pos,cal_block_id,cal_block_variant,ra,dec,obs_time_utc,exposure_time
,,,,,,,,,,deg,deg,,
object,int64,int64,object,object,object,object,object,object,object,float64,float64,object,float64


#### Catalogues

In [ ]:
catalogue_vis = da.get_catalogues_for_observation(2683, filter="VIS", dither="0-1")

In [ ]:
catalogue_vis[:5]

NUMBER,X_IMAGE,Y_IMAGE,XPSF_IMAGE,YPSF_IMAGE,X_WORLD,Y_WORLD,ERRA_WORLD,ERRB_WORLD,ERRTHETA_WORLD,XPSF_WORLD,YPSF_WORLD,ERRAPSF_WORLD,ERRBPSF_WORLD,ERRTHETAPSF_WORLD,A_WORLD,B_WORLD,THETA_WORLD,ERRX2_WORLD,ERRY2_WORLD,ALPHA_J2000,DELTA_J2000,ERRTHETA_J2000,THETA_J2000,XWIN_WORLD,YWIN_WORLD,ALPHAPSF_J2000,DELTAPSF_J2000,ERRTHETAPSF_J2000,ALPHAWIN_J2000,DELTAWIN_J2000,ERRAWIN_WORLD,ERRBWIN_WORLD,ERRTHETAWIN_WORLD,AWIN_WORLD,BWIN_WORLD,THETAWIN_WORLD,ERRX2WIN_WORLD,ERRY2WIN_WORLD,MU_THRESHOLD,MU_MAX,MU_MAX_MODEL,MU_EFF_MODEL,MU_MEAN_MODEL,MU_MAX_SPHEROID,MU_EFF_SPHEROID,MU_MEAN_SPHEROID,MU_MAX_DISK,MU_EFF_DISK,MU_MEAN_DISK,ELLIPTICITY,FLUX_AUTO,FLUXERR_AUTO,MAG_AUTO,MAGERR_AUTO,BACKGROUND,MAG_APER,MAGERR_APER,FLUX_APER,FLUXERR_APER,MAG_PSF,MAGERR_PSF,FLUX_RADIUS,ELONGATION,FWHM_IMAGE,SNR_WIN,SPREAD_MODEL,FLAGS,FLAGS_WIN,FLAGS_MODEL,ERRAWIN_IMAGE,ERRBWIN_IMAGE,ERRTHETAWIN_IMAGE,VECTOR_MODEL,VECTOR_MODELERR,MATRIX_MODELERR,CHI2_MODEL,NITER_MODEL,FLUX_MODEL,FLUXERR_MODEL,MAG_MODEL,MAGERR_MODEL,XMODEL_IMAGE,YMODEL_IMAGE,XMODEL_WORLD,YMODEL_WORLD,ELLIP1MODEL_IMAGE,ELLIP2MODEL_IMAGE,ELLIP1ERRMODEL_IMAGE,ELLIP2ERRMODEL_IMAGE,ELLIPCORRMODEL_IMAGE,ELLIP1MODEL_WORLD,ELLIP2MODEL_WORLD,ELLIP1ERRMODEL_WORLD,ELLIP2ERRMODEL_WORLD,ELLIPCORRMODEL_WORLD,IMAFLAGS_ISO,NIMAFLAGS_ISO,XWIN_IMAGE,YWIN_IMAGE,MAG_APER_RAW,MAG_AUTO_RAW,FLUX_APER_CAL,FLUXERR_APER_CAL,FLUX_AUTO_CAL,FLUXERR_AUTO_CAL,SNR,DETECTOR
,pix,pix,pix,pix,deg,deg,deg,deg,deg,deg,deg,pix,pix,deg,deg,deg,deg,deg2,deg2,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg,deg2,deg2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,mag / arcsec2,,ct,ct,mag,mag,ct,mag,mag,ct,ct,mag,mag,pix,,pix,,,,,,pix,pix,deg,,,,,,ct,ct,mag,mag,pix,pix,deg,deg,,,,,,,,,,,,,pix,pix,mag,mag,uJy,uJy,uJy,uJy,,
int32,float32,float32,float64,float64,float64,float64,float32,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float64,float64,float64,float64,float32,float32,float64,float64,float64,float64,float32,float64,float64,float32,float32,float32,float32,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32[4],float32[4],float32[4],float32[4],float32,float32,float32,float32,float32,float32,float32,int16,int16,uint8,float32,float32,float32,float32[10],float32[10],"float32[10,10]",float32,int16,float32,float32,float32,float32,float64,float64,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,int32,float64,float64,float32[4],float32,float32[4],float32[4],float32,float32,float32,str5
1,667.0519,127.9629,666.9959,126.4908,2.6808442317e+02,6.7741106086e+01,2.261546e-06,1.743684e-06,-17.51,2.6808433347e+02,6.7741082849e+01,1.211027e-06,1.138809e-06,-79.41,0.0001964726,0.0001479031,-19.12,4.9267508956e-12,3.2282754952e-12,268.0844232,67.7411061,-72.49,-70.88,2.6808439735e+02,6.7741102770e+01,268.0843335,67.7410828,-10.59,268.0843973,67.7411028,2.635276e-06,2.53961e-06,-26.98,0.0001534727,0.0001296107,-23.41,6.8427690385e-12,6.5515334934e-12,29.8478,26.7567,26.9720,29.7759,28.5459,26.7205,35.0473,33.6544,26.9143,28.7365,28.0387,0.394,10941.67,153.5101,21.2902,0.0152,23.14396,23.4905 .. 21.3644,0.0218 .. 0.0144,1441.932 .. 10218.57,28.90709 .. 135.6537,24.4018,0.0303,9.112,1.650,22.81,106.6,0.029957,2,0,0,0.09558,0.08971,-42.27,-0.04711 .. -45.32,0.07781 .. 1.046,0.006055 .. 1.095,0.9294773,66,15301.29,2515.29,20.9261,0.1785,666.9529,127.6304,2.6808440627e+02,6.7741098792e+01,-0.027190,0.069484,0.045421,0.116359,-0.980679,-0.021327,0.056198,0.033016,0.090783,-0.965327,16777235,627,667.1381,127.6029,-7.8973618 .. -10.023476,-10.09771,1.4580983 .. 10.333136,0.029231172 .. 0.13717453,11.064341,0.15523113,75.328384,1-1.E
2,1615.5247,90.2938,1615.1601,89.7841,2.6804510633e+02,6.7763004102e+01,1.813975e-06,1.227406e-06,43.71,

In [ ]:
catalogues_nir = da.get_catalogues_for_observation(2683, instrument="NISP")

In [ ]:
catalogues_nir["NIR_H"]["1"][:5]

OBJECT_ID,XWIN_IMAGE,YWIN_IMAGE,XWIN_WORLD,YWIN_WORLD,ALPHA_J2000,DELTA_J2000,MAG_APER,MAGERR_APER,FLUX_APER,FLUXERR_APER,MAG_AUTO,MAGERR_AUTO,FLUX_AUTO,FLUXERR_AUTO,AWIN_WORLD,BWIN_WORLD,THETAWIN_WORLD,ERRAWIN_WORLD,ERRBWIN_WORLD,ERRTHETAWIN_WORLD,KRON_RADIUS,ELLIPTICITY,CLASS_STAR,FWHM_IMAGE,FLUX_RADIUS,BACKGROUND,THRESHOLD,XPSF_IMAGE,YPSF_IMAGE,XPSF_WORLD,YPSF_WORLD,ALPHAPSF_J2000,DELTAPSF_J2000,MAG_PSF,MAGERR_PSF,FLUX_PSF,FLUXERR_PSF,ERRAPSF_IMAGE,ERRBPSF_IMAGE,ERRTHETAPSF_IMAGE,CHI2_PSF,FLAGS,DETECTOR
,pix,pix,deg,deg,deg,deg,mag,mag,ct,ct,mag,mag,ct,ct,deg,deg,deg,deg,deg,deg,,,,pix,pix,ct,ct,pix,pix,deg,deg,deg,deg,mag,mag,ct,ct,pix,pix,deg,,,
int32,float64,float64,float64,float64,float64,float64,float32[5],float32[5],float32[5],float32[5],float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,float64,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,int16,str2
21,50.5984,12.6080,2.6992668317e+02,6.8131641471e+01,269.9267695,68.1316032,24.7452 .. 23.3146,0.3059 .. 0.4023,128.6682 .. 480.5274,36.2375 .. 178.0027,24.5335,0.2846,156.371,40.97853,5.389608e-05,4.574121e-05,-67.08,2.189676e-05,2.162293e-05,-77.89,3.50,0.355,0.353,2.87,1.050,68.98931,0.1208089,50.0000,12.5000,2.6992676951e+02,6.8131603221e+01,269.9267695,68.1316032,99.0000,99.0000,0,0,0.00000,0.00000,0.00,0,0,11
24,328.8609,10.4722,2.6989644067e+02,6.8151595219e+01,269.8964916,68.1515914,24.3336 .. 20.9324,0.2129 .. 0.0461,187.9903 .. 4311.135,36.86229 .. 183.1946,24.2323,0.2836,206.3592,53.89828,5.319995e-05,5.235874e-05,-48.71,1.77482e-05,1.760394e-05,36.07,3.50,0.188,0.326,3.93,1.054,63.82212,0.1208089,328.7049,10.2949,2.6989649160e+02,6.8151591407e+01,269.8964916,68.1515914,99.0000,99.0000,0,0,0.00000,0.00000,0.00,0,0,11
25,401.5988,10.4432,2.6988841971e+02,6.8156790518e+01,269.8884198,68.1567811,23.3792 .. 23.1454,0.0926 .. 0.3445,452.7687 .. 561.5798,38.61628 .. 178.1615,23.0551,0.1054,610.2734,59.222,6.524716e-05,5.817264e-05,-45.68,9.560273e-06,9.474173e-06,-60.29,3.50,0.000,0.941,2.13,1.295,62.89843,0.1208089,401.5000,10.4999,2.6988841983e+02,6.8156781121e+01,269.8884198,68.1567811,99.0000,99.0000,0,0,0.00000,0.00000,0.00,0,0,11
26,1679.2416,11.2159,2.6974630700e+02,6.8248054931e+01,269.7463915,68.2479544,21.9470 .. 20.4114,0.0294 .. 0.0291,1693.342 .. 6966.458,45.77656 .. 186.8902,21.7281,0.0269,2071.624,51.35043,5.387569e-05,5.351707e-05,59.35,2.39664e-06,2.376038e-06,-84.50,3.50,0.354,0.935,3.67,1.387,63.65472,0.1208089,1678.0000,11.5000,2.6974639150e+02,6.8247954388e+01,269.7463915,68.2479544,99.0000,99.0000,0,0,0.00000,0.00000,0.00,0,0,11
27,1979.5024,10.9728,2.6971272992e+02,6.8269532079e+01,269.7123465,68.2698011,23.6846 .. 18.7390,0.1201 .. 0.0073,341.7659 .. 32504.41,37.78506 .. 218.3321,23.3699,0.1097,456.6743,46.14747,4.24568e-05,1.79031e-05,-59.80,4.411954e-07,2.809796e-07,-59.89,3.50,0.163,0.424,4.37,1.427,62.79366,0.1208089,1983.1775,10.8226,2.6971234655e+02,6.8269801079e+01,269.7123465,68.2698011,99.0000,99.0000,0,0,0.00000,0.00000,0.00,0,0,11


#### Async queries

In [ ]:
query = "SELECT TOP 1 cat.object_id, phz.object_id FROM catalogue.mer_catalogue_wide_survey as cat JOIN catalogue.phz_photo_z_wide as phz USING (object_id)"
da.tap_query(query, async_only=True)

2026-02-27 11:13:41 - __main__._tap_query_async_wait - INFO - TAP async job still running (phase=EXECUTING). Next poll in 5s at 2026-02-27 11:13:46.
2026-02-27 11:13:46 - __main__._tap_query_async_wait - INFO - TAP async job still running (phase=EXECUTING). Next poll in 10s at 2026-02-27 11:13:56.
2026-02-27 11:13:56 - __main__._tap_query_async_wait - INFO - TAP async job still running (phase=EXECUTING). Next poll in 15s at 2026-02-27 11:14:11.
2026-02-27 11:14:11 - __main__._tap_query_async_wait - INFO - TAP async job still running (phase=EXECUTING). Next poll in 20s at 2026-02-27 11:14:31.


object_id,object_id_2
int64,int64
1896880493667911401,1896880493667911401
